## Walmart-ის გაყიდვების პროგნოზირება — LightGBM

| Field | Details |
|---|---|
| **მოდელი** | LightGBM Regressor |
| **კატეგორია** | Tree-based — Gradient Boosting, leaf-wise growth |
| **MLflow ექსპერიმენტი** | `LightGBM_Training` |
| **მეტრიკა** | WMAE — სადღესასწაულო კვირებს ენიჭებათ 5-მაგი წონა |

---

### Runs

| Run | აღწერა |
|---|---|
| `LightGBM_Cleaning` | Preprocessing pipeline-ის ვალიდაცია |
| `LightGBM_Feature_Selection` | Feature ნაკრებების შედარება (lag/rolling ფიჩერების ჩათვლით) |
| `LightGBM_CrossValidation` | ქროს ვალიდაცია — TimeSeriesSplit CV |
| `LightGBM_GridSearch` | ჰიპერპარამეტრების Grid Search — 20 კომბინაცია |
| `LightGBM_Final` | საუკეთესო pipeline (raw მონაცემებიდან prediction-მდე) |

---


## 1. Setup

In [ ]:
!pip install lightgbm mlflow dagshub scikit-learn --quiet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid

import lightgbm as lgb
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

print(f"LightGBM version: {lgb.__version__}")
print(f"MLflow version: {mlflow.__version__}")


In [ ]:
def safe_log1p(y):
    return np.log1p(np.clip(np.asarray(y, dtype=float), a_min=0, a_max=None))


class LogTargetLGBMRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, **lgb_params):
        self.lgb_params = lgb_params

    def fit(self, X, y, eval_set=None, **fit_params):
        y_log = safe_log1p(y)
        self.model_ = lgb.LGBMRegressor(**self.lgb_params)
        if eval_set is not None:
            eval_set_log = [(ex, safe_log1p(ey)) for ex, ey in eval_set]
            self.model_.fit(X, y_log, eval_set=eval_set_log, **fit_params)
        else:
            self.model_.fit(X, y_log, **fit_params)
        self.best_iteration_ = getattr(self.model_, 'best_iteration_', None)
        self.feature_importances_ = self.model_.feature_importances_
        return self

    def predict(self, X, num_iteration=None):
        if num_iteration is not None:
            pred_log = self.model_.predict(X, num_iteration=num_iteration)
        else:
            pred_log = self.model_.predict(X)
        return np.expm1(pred_log)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
MODELS_DIR = f'{PROJECT_DIR}/models'

import os
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")


In [ ]:
import dagshub

DAGSHUB_USERNAME = "zberi23"
DAGSHUB_REPO = "walmart-forecasting"

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

EXPERIMENT_NAME = "LightGBM_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")


## 2. Raw Data Loading

Pipeline იმუშავებს raw test set-ზე. სამივე original ფაილს ვტვირთავთ.

In [ ]:
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
stores_raw = pd.read_csv(f'{DATA_DIR}/stores.csv')
features_raw = pd.read_csv(f'{DATA_DIR}/features.csv.zip')

for df in [train_raw, test_raw, features_raw]:
    df['Date'] = pd.to_datetime(df['Date'])

print(f"Train raw: {train_raw.shape}")
print(f"Test raw:  {test_raw.shape}")
print(f"Stores:    {stores_raw.shape}")
print(f"Features:  {features_raw.shape}")

train_raw.head(3)


## 3. Exploratory Data Analysis

### 3.1 საერთო სტრუქტურა და აკლია მონაცემები

In [ ]:
print("train_raw info:")
train_raw.info()
print("\nfeatures_raw missing values:")
print(features_raw.isnull().sum())
print("\ntrain_raw duplicate rows:", train_raw.duplicated(subset=['Store', 'Dept', 'Date']).sum())
print("\nDate range train:", train_raw['Date'].min(), "->", train_raw['Date'].max())
print("Date range test: ", test_raw['Date'].min(), "->", test_raw['Date'].max())
print("\nUnique stores:", train_raw['Store'].nunique(), "| Unique depts:", train_raw['Dept'].nunique())


In [ ]:
train_raw['Weekly_Sales'].describe()


### 3.2 გაყიდვების განაწილება

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train_raw['Weekly_Sales'], bins=100, color='steelblue')
axes[0].set_title('Weekly_Sales განაწილება')
axes[0].set_xlabel('Weekly_Sales')

negative_share = (train_raw['Weekly_Sales'] < 0).mean() * 100
axes[1].hist(np.clip(train_raw['Weekly_Sales'], -5000, 30000), bins=100, color='indianred')
axes[1].set_title(f'გადაკვეთილი დიაპაზონი (უარყოფითი: {negative_share:.2f}%)')
plt.tight_layout()
plt.show()


### 3.3 დროში ტენდენცია და დღესასწაულების ეფექტი

In [ ]:
weekly_totals = train_raw.groupby('Date')['Weekly_Sales'].sum()
plt.figure(figsize=(14, 4))
plt.plot(weekly_totals.index, weekly_totals.values)
plt.title('ჯამური კვირეული გაყიდვები დროში')
plt.xlabel('Date')
plt.ylabel('Total Weekly_Sales')
plt.show()

holiday_avg = train_raw.groupby('IsHoliday')['Weekly_Sales'].mean()
print("საშუალო გაყიდვები დღესასწაულზე vs ჩვეულებრივ კვირაზე:")
print(holiday_avg)


### 3.4 მაღაზიის ტიპი და ზომა

In [ ]:
store_info = train_raw.merge(stores_raw, on='Store', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
store_info.groupby('Type')['Weekly_Sales'].mean().plot(kind='bar', ax=axes[0], color='seagreen')
axes[0].set_title('საშუალო გაყიდვები Store Type მიხედვით')

axes[1].scatter(stores_raw['Size'], stores_raw.groupby('Store').ngroup(), s=10)
axes[1].set_title('Store Size განაწილება')
axes[1].set_xlabel('Size')
plt.tight_layout()
plt.show()

print(stores_raw.groupby('Type')['Size'].describe())


### 3.5 დეპარტამენტების მიხედვით ცვალებადობა

In [ ]:
dept_stats = train_raw.groupby('Dept')['Weekly_Sales'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
print("ტოპ 10 დეპარტამენტი საშუალო გაყიდვებით:")
print(dept_stats.head(10))
print("\nყველაზე ცვალებადი 10 დეპარტამენტი (std):")
print(dept_stats.sort_values('std', ascending=False).head(10))


### 3.6 კორელაცია რიცხვით ცვლადებთან

In [ ]:
merged_for_corr = train_raw.merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
corr_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
corr = merged_for_corr[corr_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('კორელაცია Weekly_Sales-სთან')
plt.tight_layout()
plt.show()


**EDA დასკვნა:** გაყიდვები ძლიერად სეგმენტირებულია Store/Dept მიხედვით, დღესასწაულებზე საშუალო გაყიდვები აღემატება ჩვეულებრივ კვირებს, ეკონომიკური ცვლადები (Temperature, Fuel_Price, CPI, Unemployment) სუსტად უკავშირდება target-ს პირდაპირ, ამიტომ Store/Dept-ის ისტორიული ქცევის ამსახველი lag და rolling ფიჩერები დიდი მნიშვნელობის მატარებელი უნდა იყოს.

## 4. Lag და Rolling ფიჩერები

Lag/rolling მნიშვნელობები გამოთვლილია Store×Dept ჯგუფის მიხედვით, დროში დალაგებულ მონაცემებზე. `test_raw`-ს არ გააჩნია `Weekly_Sales`, ამიტომ ის NaN-ით ემატება ერთობლივ სერიას — `shift()` ვერასდროს წაიღებს მომავალი (ჯერ არცნობილი) მნიშვნელობას, ის მხოლოდ წარსულ, უკვე დაფიქსირებულ გაყიდვებზე დაყრდნობით ითვლის ფიჩერებს, რაც გამორიცხავს data leakage-ს.

**გასათვალისწინებელი დეტალი:** ვინაიდან train მხოლოდ ~143 კვირას მოიცავს, ყველაზე გრძელი lag-ები (`364`-ზე ზემოთ) ხშირად NaN გამოვა და fallback-ით შეივსება — ეს გამჭვირვალედ ჩანს ქვემოთ დაბეჭდილ NaN დათვლაში.

NaN-ების fallback შევსება ხდება მხოლოდ `shift(1).expanding().median()`-ით — ე.ი. მხოლოდ იმ Store×Dept-ის მანამდელი, უკვე დაფიქსირებული გაყიდვებით. ადრინდელი ვერსია იყენებდა `groupby(...).transform('median')`-ს მთელი ერთობლივი (train+test) სერიის მიხედვით, რაც ადრეულ რიგებს ავსებდა მომავალი, ჯერ არდაფიქსირებული პერიოდის მონაცემებით — ეს იყო რეალური data leakage და ამან ხელოვნურად დაწია WMAE.

In [ ]:
LAG_WEEKS = [91, 98, 105, 112, 119, 126, 182, 364, 546, 728]
ROLL_WINDOWS = [365, 546]
ROLL_MAX_WINDOWS = [4]


def random_noise(dataframe):
    return np.random.normal(scale=1.5, size=(len(dataframe),))


def build_lag_rolling_features(train_df, test_df, lags=LAG_WEEKS, windows=ROLL_WINDOWS, max_windows=ROLL_MAX_WINDOWS):
    hist = train_df[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
    fut = test_df[['Store', 'Dept', 'Date']].copy()
    fut['Weekly_Sales'] = np.nan

    combined = pd.concat([hist, fut], axis=0, ignore_index=True)
    combined = combined.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    grouped_sales = combined.groupby(['Store', 'Dept'])['Weekly_Sales']

    for lag in lags:
        combined[f'sales_lag_{lag}'] = grouped_sales.transform(lambda s: s.shift(lag)) + random_noise(combined)

    for window in windows:
        combined[f'sales_roll_mean_{window}'] = combined.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=10, win_type='triang').mean()
        ) + random_noise(combined)

    for window in max_windows:
        combined[f'roll_max_{window}w'] = combined.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(
            lambda s: s.shift(1).rolling(window=window, min_periods=1).max()
        )

    feature_cols = [c for c in combined.columns if c.startswith('sales_lag_') or c.startswith('sales_roll_') or c.startswith('roll_max_')]

    nan_before_fallback = combined[feature_cols].isnull().sum()
    print("NaN count per lag/rolling column before fallback fill:")
    print(nan_before_fallback[nan_before_fallback > 0])

    combined['_causal_fallback'] = combined.groupby(['Store', 'Dept'])['Weekly_Sales'].transform(
        lambda s: s.shift(1).expanding(min_periods=1).median()
    )

    for col in feature_cols:
        combined[col] = combined[col].fillna(combined['_causal_fallback'])
        combined[col] = combined[col].fillna(0)

    combined = combined.drop(columns=['_causal_fallback'])

    return combined[['Store', 'Dept', 'Date'] + feature_cols], feature_cols


LAG_DF, LAG_FEATURE_COLS = build_lag_rolling_features(train_raw, test_raw)
print(f"\nLag/rolling feature table: {LAG_DF.shape}")
print(f"Generated columns: {LAG_FEATURE_COLS}")
print(f"Remaining NaNs after fallback fill: {LAG_DF[LAG_FEATURE_COLS].isnull().sum().sum()}")


## 5. Custom Preprocessor

ერთი-ცხელი (one-hot) კოდირება ემატება ოთხ ველზე — `Store`, `Dept`, `Day_of_Week` და `Month`. კატეგორიები წინასწარაა დაფიქსირებული (`ALL_STORES`, `ALL_DEPTS`) რომ train/val/test-ს შორის სვეტები ერთმანეთს ემთხვეოდეს.

In [ ]:
ALL_STORES = sorted(stores_raw['Store'].unique().tolist())
ALL_DEPTS = sorted(pd.concat([train_raw['Dept'], test_raw['Dept']]).unique().tolist())
ALL_DOW = list(range(7))
ALL_MONTHS = list(range(1, 13))


In [ ]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """Raw Walmart CSV -> feature matrix. Ასევე ერწყმის lag/rolling და one-hot ფიჩერებს."""

    def __init__(self, stores_df, features_df, lag_df, all_stores, all_depts, all_dow, all_months):
        self.stores_df = stores_df.drop_duplicates(subset=['Store']).copy()
        self.features_df = features_df.drop_duplicates(subset=['Store', 'Date', 'IsHoliday']).copy()
        self.lag_df = lag_df.drop_duplicates(subset=['Store', 'Dept', 'Date']).copy()
        self.all_stores = all_stores
        self.all_depts = all_depts
        self.all_dow = all_dow
        self.all_months = all_months
        if not pd.api.types.is_datetime64_any_dtype(self.features_df['Date']):
            self.features_df['Date'] = pd.to_datetime(self.features_df['Date'])
        if not pd.api.types.is_datetime64_any_dtype(self.lag_df['Date']):
            self.lag_df['Date'] = pd.to_datetime(self.lag_df['Date'])

    def fit(self, X, y=None):
        df = X.copy().reset_index(drop=True)
        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])
        merged = df.merge(self.features_df, on=['Store', 'Date', 'IsHoliday'], how='left')
        for col in ['CPI', 'Unemployment']:
            merged[col] = merged.groupby('Store')[col].transform(lambda x: x.ffill())
        self.cpi_median_ = merged['CPI'].median()
        self.unemployment_median_ = merged['Unemployment'].median()
        if pd.isna(self.cpi_median_):
            self.cpi_median_ = 0.0
        if pd.isna(self.unemployment_median_):
            self.unemployment_median_ = 0.0

        if y is not None:
            target_df = df[['Store', 'Dept']].copy()
            target_df['_y'] = np.asarray(y)
            self.store_dept_median_ = (
                target_df.groupby(['Store', 'Dept'])['_y'].median()
                .reset_index().rename(columns={'_y': 'store_dept_median'})
            )
            self.global_median_ = float(target_df['_y'].median())
        else:
            self.store_dept_median_ = pd.DataFrame(columns=['Store', 'Dept', 'store_dept_median'])
            self.global_median_ = 0.0
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop=True)
        n_input_rows = len(df)
        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        df = df.merge(self.features_df, on=['Store', 'Date', 'IsHoliday'], how='left')
        assert len(df) == n_input_rows, (
            f"Row count changed after merging features_df: {n_input_rows} -> {len(df)}. "
            "Check features.csv for duplicate (Store, Date, IsHoliday) rows."
        )

        df = df.merge(self.stores_df, on='Store', how='left')
        assert len(df) == n_input_rows, (
            f"Row count changed after merging stores_df: {n_input_rows} -> {len(df)}. "
            "Check stores.csv for duplicate Store rows."
        )

        df = df.merge(self.lag_df, on=['Store', 'Dept', 'Date'], how='left')
        assert len(df) == n_input_rows, (
            f"Row count changed after merging lag_df: {n_input_rows} -> {len(df)}. "
            "Check LAG_DF for duplicate (Store, Dept, Date) rows."
        )

        for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
            df[col] = df[col].fillna(0)
        for col in ['CPI', 'Unemployment']:
            df[col] = df.groupby('Store')[col].transform(lambda x: x.ffill())
        df['CPI'] = df['CPI'].fillna(self.cpi_median_)
        df['Unemployment'] = df['Unemployment'].fillna(self.unemployment_median_)

        lag_cols = [c for c in self.lag_df.columns if c not in ['Store', 'Dept', 'Date']]
        for col in lag_cols:
            df[col] = df[col].fillna(0)

        df = df.merge(self.store_dept_median_, on=['Store', 'Dept'], how='left')
        df['store_dept_median'] = df['store_dept_median'].fillna(self.global_median_)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['Day_of_Year'] = df['Date'].dt.dayofyear
        df['Quarter'] = df['Date'].dt.quarter
        df['Day_of_Week'] = df['Date'].dt.dayofweek

        df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
        df['Week_Sin'] = np.sin(2 * np.pi * df['Week'] / 52)
        df['Week_Cos'] = np.cos(2 * np.pi * df['Week'] / 52)

        super_bowl = pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'])
        labor_day = pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'])
        thanksgiving = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'])
        christmas = pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])

        df['Is_SuperBowl'] = df['Date'].isin(super_bowl).astype(int)
        df['Is_LaborDay'] = df['Date'].isin(labor_day).astype(int)
        df['Is_Thanksgiving'] = df['Date'].isin(thanksgiving).astype(int)
        df['Is_Christmas'] = df['Date'].isin(christmas).astype(int)

        df['Type_Encoded'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)
        df = df.drop(columns=['Type'])

        store_dummies = pd.get_dummies(
            pd.Categorical(df['Store'], categories=self.all_stores), prefix='store'
        ).astype(int)
        dept_dummies = pd.get_dummies(
            pd.Categorical(df['Dept'], categories=self.all_depts), prefix='dept'
        ).astype(int)
        dow_dummies = pd.get_dummies(
            pd.Categorical(df['Day_of_Week'], categories=self.all_dow), prefix='dow'
        ).astype(int)
        month_dummies = pd.get_dummies(
            pd.Categorical(df['Month'], categories=self.all_months), prefix='month'
        ).astype(int)

        store_dummies.index = df.index
        dept_dummies.index = df.index
        dow_dummies.index = df.index
        month_dummies.index = df.index

        df = pd.concat([df, store_dummies, dept_dummies, dow_dummies, month_dummies], axis=1)

        df = df.drop(columns=['Date'])
        return df


ONEHOT_COLS = (
    [f'store_{s}' for s in ALL_STORES] + [f'dept_{d}' for d in ALL_DEPTS]
    + [f'dow_{d}' for d in ALL_DOW] + [f'month_{m}' for m in ALL_MONTHS]
)

preprocessor = WalmartPreprocessor(stores_raw, features_raw, LAG_DF, ALL_STORES, ALL_DEPTS, ALL_DOW, ALL_MONTHS)


## 6. WMAE Metric + Train/Val Split

In [ ]:
def wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def get_weights(is_holiday):
    return np.where(is_holiday == 1, 5, 1)


train_sorted = train_raw.sort_values('Date').reset_index(drop=True)
split_date = train_sorted['Date'].quantile(0.9)

X_train_raw = train_sorted[train_sorted['Date'] < split_date].drop(columns=['Weekly_Sales'])
y_train = train_sorted[train_sorted['Date'] < split_date]['Weekly_Sales'].values

X_val_raw = train_sorted[train_sorted['Date'] >= split_date].drop(columns=['Weekly_Sales'])
y_val = train_sorted[train_sorted['Date'] >= split_date]['Weekly_Sales'].values

print(f"Train: {X_train_raw.shape}, Val: {X_val_raw.shape}")
print(f"Train dates: {X_train_raw['Date'].min()} -> {X_train_raw['Date'].max()}")
print(f"Val dates:   {X_val_raw['Date'].min()} -> {X_val_raw['Date'].max()}")


## 7. Run 1 — `LightGBM_Cleaning`

Preprocessing validation.

In [ ]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    X_train_processed = preprocessor.fit_transform(X_train_raw, y_train)
    X_val_processed = preprocessor.transform(X_val_raw)
    X_test_processed = preprocessor.transform(test_raw)

    train_nans = X_train_processed.isnull().sum().sum()
    val_nans = X_val_processed.isnull().sum().sum()
    test_nans = X_test_processed.isnull().sum().sum()

    mlflow.log_param("train_rows_raw", X_train_raw.shape[0])
    mlflow.log_param("train_rows_processed", X_train_processed.shape[0])
    mlflow.log_param("test_rows_processed", X_test_processed.shape[0])
    mlflow.log_param("n_features_after_preprocessing", X_train_processed.shape[1])
    mlflow.log_param("markdown_fill_value", 0)
    mlflow.log_param("cpi_unemployment_fill_method", "groupby_store_ffill_bfill")
    mlflow.log_param("lag_weeks", LAG_WEEKS)
    mlflow.log_param("rolling_windows", ROLL_WINDOWS)
    mlflow.log_param("features", list(X_train_processed.columns))

    mlflow.log_metric("train_missing_after_clean", train_nans)
    mlflow.log_metric("val_missing_after_clean", val_nans)
    mlflow.log_metric("test_missing_after_clean", test_nans)

    print(f"Train NaN: {train_nans}, Val NaN: {val_nans}, Test NaN: {test_nans}")
    print(f"Features: {X_train_processed.shape[1]}")

    mlflow.set_tag("stage", "cleaning")


## 8. Run 2 — `LightGBM_Feature_Selection`

ოთხი feature set-ის შედარება baseline LightGBM-ით.

In [ ]:
CORE_FEATURES = ['Store', 'Dept', 'IsHoliday', 'Size', 'Type_Encoded',
                  'Year', 'Month', 'Week', 'Day_of_Year', 'store_dept_median',
                  'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
                  'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos']

EXTRA_ENCODED_FEATURES = (
    ['IsHoliday', 'Size', 'Type_Encoded', 'Quarter', 'Week', 'Day_of_Year', 'store_dept_median',
     'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas']
    + ONEHOT_COLS
)

FEATURE_SETS = {
    "all": list(X_train_processed.columns),
    "no_markdown": [c for c in X_train_processed.columns
                    if c not in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']],
    "core": CORE_FEATURES,
    "core_plus_lag_rolling": CORE_FEATURES + LAG_FEATURE_COLS,
    "lag_rolling_onehot": EXTRA_ENCODED_FEATURES + LAG_FEATURE_COLS,
}

BASELINE_PARAMS = {
    'n_estimators': 200,
    'max_depth': 8,
    'num_leaves': 63,
    'learning_rate': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

feature_selection_results = {}

with mlflow.start_run(run_name="LightGBM_Feature_Selection"):
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_param("feature_sets_tested", list(FEATURE_SETS.keys()))

    for fs_name, feats in FEATURE_SETS.items():
        with mlflow.start_run(run_name=f"FS_{fs_name}", nested=True):
            X_tr = X_train_processed[feats]
            X_va = X_val_processed[feats]

            model = LogTargetLGBMRegressor(**BASELINE_PARAMS)
            model.fit(X_tr, y_train, eval_set=[(X_va, y_val)],
                      callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

            preds = model.predict(X_va)
            weights = get_weights(X_val_processed['IsHoliday'].values)
            score = wmae(y_val, preds, weights)

            feature_selection_results[fs_name] = score
            mlflow.log_param("feature_set", fs_name)
            mlflow.log_param("n_features", len(feats))
            mlflow.log_metric("val_wmae", score)

            print(f"[{fs_name:22}] n_features={len(feats):3d}, WMAE = {score:.2f}")

    ranked_fs = sorted(feature_selection_results, key=feature_selection_results.get)
    for rank, name in enumerate(ranked_fs, 1):
        print(f"  #{rank} {name}: WMAE = {feature_selection_results[name]:.2f}")

    best_fs = "lag_rolling_onehot"
    BEST_FEATURES = FEATURE_SETS[best_fs]
    mlflow.log_metric("best_val_wmae", feature_selection_results[best_fs])
    mlflow.log_param("best_feature_set", best_fs)
    mlflow.log_param("feature_set_selection_rule", "forced_lag_rolling")
    mlflow.set_tag("stage", "feature_selection")

print(f"\nSelected feature set: {best_fs} ({len(BEST_FEATURES)} features)")


## 9. Run 3 — `LightGBM_CrossValidation`

TimeSeriesSplit — 5 folds.

In [ ]:
N_SPLITS = 5
X_train_selected = X_train_processed[BEST_FEATURES]

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_scores = []

with mlflow.start_run(run_name="LightGBM_CrossValidation"):
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_param("cv_method", "TimeSeriesSplit")
    mlflow.log_param("n_splits", N_SPLITS)
    mlflow.log_param("feature_set", best_fs)

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_selected)):
        X_tr_fold = X_train_selected.iloc[tr_idx]
        y_tr_fold = y_train[tr_idx]
        X_va_fold = X_train_selected.iloc[va_idx]
        y_va_fold = y_train[va_idx]

        model = LogTargetLGBMRegressor(**BASELINE_PARAMS)
        model.fit(X_tr_fold, y_tr_fold, callbacks=[lgb.log_evaluation(0)])

        preds = model.predict(X_va_fold)
        weights_fold = get_weights(X_train_processed['IsHoliday'].iloc[va_idx].values)
        score = wmae(y_va_fold, preds, weights_fold)

        cv_scores.append(score)
        mlflow.log_metric("fold_wmae", score, step=fold)
        print(f"Fold {fold + 1}: WMAE = {score:.2f}")

    mean_score = np.mean(cv_scores)
    std_score = np.std(cv_scores)
    mlflow.log_metric("cv_wmae_mean", mean_score)
    mlflow.log_metric("cv_wmae_std", std_score)
    mlflow.set_tag("stage", "cross_validation")

    print(f"\nCV WMAE: {mean_score:.2f} +/- {std_score:.2f}")


## 10. Run 4 — `LightGBM_GridSearch`

20-კომბინაციიანი Grid Search `num_leaves` x `learning_rate`-ზე. მნიშვნელობები ცენტრირებულია საკმაოდ მაღალ `num_leaves`-სა (LightGBM-ის leaf-wise ზრდისთვის `num_leaves` არის მთავარი რეგულარიზაციის პარამეტრი, არა `max_depth`) და ზომიერ `learning_rate`-ზე. `max_depth=-1` (შეუზღუდავი) რჩება ფიქსირებული, რადგან `num_leaves` თავად საკმარისად აკონტროლებს სირთულეს.

**Grid:**
- `num_leaves`: [95, 127, 187, 255]
- `learning_rate`: [0.03, 0.04, 0.05, 0.07, 0.09]
- ფიქსირებული: `max_depth=-1`, `subsample=0.77`, `colsample_bytree=0.55`, `min_child_samples=60`, `reg_alpha=0.6`, `reg_lambda=2.5`


In [ ]:
PARAM_GRID = {
    'num_leaves': [95, 127, 187, 255],
    'learning_rate': [0.03, 0.04, 0.05, 0.07, 0.09],
}
FIXED_PARAMS = {
    'n_estimators': 3000,
    'max_depth': -1,
    'subsample': 0.77,
    'colsample_bytree': 0.55,
    'min_child_samples': 60,
    'reg_alpha': 0.6,
    'reg_lambda': 2.5,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

grid_combinations = list(ParameterGrid(PARAM_GRID))
print(f"Total grid iterations: {len(grid_combinations)}")

grid_tscv = TimeSeriesSplit(n_splits=3)
grid_results = []

with mlflow.start_run(run_name="LightGBM_GridSearch"):
    mlflow.log_param("search_method", "GridSearchCV_manual")
    mlflow.log_param("n_iterations", len(grid_combinations))
    mlflow.log_param("cv_splits", grid_tscv.n_splits)
    mlflow.log_param("feature_set", best_fs)
    mlflow.log_params(FIXED_PARAMS)

    for i, combo in enumerate(grid_combinations):
        params = {**FIXED_PARAMS, **combo}
        fold_scores = []

        for tr_idx, va_idx in grid_tscv.split(X_train_selected):
            X_tr_fold = X_train_selected.iloc[tr_idx]
            y_tr_fold = y_train[tr_idx]
            X_va_fold = X_train_selected.iloc[va_idx]
            y_va_fold = y_train[va_idx]

            model = LogTargetLGBMRegressor(**params)
            model.fit(X_tr_fold, y_tr_fold, eval_set=[(X_va_fold, y_va_fold)],
                      callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
            preds = model.predict(X_va_fold, num_iteration=model.best_iteration_)
            weights_fold = get_weights(X_train_processed['IsHoliday'].iloc[va_idx].values)
            fold_scores.append(wmae(y_va_fold, preds, weights_fold))

        mean_wmae = float(np.mean(fold_scores))
        grid_results.append({**combo, "mean_wmae": mean_wmae})

        with mlflow.start_run(run_name=f"grid_{i:02d}", nested=True):
            mlflow.log_params(combo)
            mlflow.log_metric("cv_wmae_mean", mean_wmae)

        print(f"[{i + 1:2d}/{len(grid_combinations)}] {combo} -> WMAE = {mean_wmae:.2f}")

    grid_results_df = pd.DataFrame(grid_results).sort_values("mean_wmae").reset_index(drop=True)
    best_combo = grid_results_df.iloc[0].to_dict()
    best_wmae_grid = best_combo.pop("mean_wmae")

    BEST_PARAMS = {**FIXED_PARAMS, **{k: (int(v) if k == 'num_leaves' else float(v)) for k, v in best_combo.items()}}

    mlflow.log_params(BEST_PARAMS)
    mlflow.log_metric("best_cv_wmae", best_wmae_grid)
    mlflow.set_tag("stage", "grid_search")

    print(f"\nBest CV WMAE: {best_wmae_grid:.2f}")
    print(f"Best params: {BEST_PARAMS}")


## 11. Run 4b — ოპტიმალური `n_estimators` (Early Stopping)

Grid search-ის შიდა early stopping-ს (`early_stopping_rounds=200`) თითოეულ fold-ზე თავისი `best_iteration_` აქვს. აქ საბოლოო ვალიდაციაზე (`X_val_processed`/`y_val`) ერთხელ კიდევ ვადგენთ ზუსტ საბოლოო `n_estimators`-ს, იმავე `early_stopping_rounds=200`-ით, სანამ სრულ train+val მონაცემზე საბოლოო მოდელს გავწვრთნით.

In [ ]:
with mlflow.start_run(run_name="LightGBM_EarlyStopping_Tuning"):
    tuning_params = {**BEST_PARAMS, 'n_estimators': 10000}
    tuning_model = LogTargetLGBMRegressor(**tuning_params)
    tuning_model.fit(
        X_train_selected, y_train,
        eval_set=[(X_val_processed[BEST_FEATURES], y_val)],
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
    )

    optimal_n_estimators = max(tuning_model.best_iteration_, 100)
    BEST_PARAMS['n_estimators'] = optimal_n_estimators

    tuned_preds = tuning_model.predict(X_val_processed[BEST_FEATURES], num_iteration=tuning_model.best_iteration_)
    tuned_weights = get_weights(X_val_processed['IsHoliday'].values)
    tuned_wmae = wmae(y_val, tuned_preds, tuned_weights)

    mlflow.log_params(BEST_PARAMS)
    mlflow.log_metric("optimal_n_estimators", optimal_n_estimators)
    mlflow.log_metric("val_wmae", tuned_wmae)
    mlflow.set_tag("stage", "early_stopping_tuning")

    print(f"Optimal n_estimators: {optimal_n_estimators}")
    print(f"Val WMAE with tuned n_estimators: {tuned_wmae:.2f}")
    print(f"Final BEST_PARAMS: {BEST_PARAMS}")


## 12. Run 5 — `LightGBM_Final` (Pipeline)

საბოლოო Pipeline:
```
raw train/test csv (Store, Dept, Date, IsHoliday)
         v
WalmartPreprocessor (merge + lag/rolling + date features)
         v
FeatureSelector
         v
LightGBM (grid search-ით მოძებნილი პარამეტრები)
         v
predictions
```

In [ ]:
class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, features):
        self.features = features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.features]


X_full_raw = train_sorted.drop(columns=['Weekly_Sales'])
y_full = train_sorted['Weekly_Sales'].values

final_pipeline = Pipeline([
    ('preprocess', WalmartPreprocessor(stores_raw, features_raw, LAG_DF, ALL_STORES, ALL_DEPTS, ALL_DOW, ALL_MONTHS)),
    ('select', FeatureSelector(BEST_FEATURES)),
    ('model', LogTargetLGBMRegressor(**BEST_PARAMS)),
])

print("Pipeline steps:")
for name, step in final_pipeline.steps:
    print(f"  {name}: {type(step).__name__}")


In [ ]:
with mlflow.start_run(run_name="LightGBM_Final") as final_run:
    final_pipeline.fit(X_full_raw, y_full)

    val_preds = final_pipeline.predict(X_val_raw)
    val_weights = get_weights(preprocessor.transform(X_val_raw)['IsHoliday'].values)
    val_wmae = wmae(y_val, val_preds, val_weights)

    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("pipeline_steps", str([s[0] for s in final_pipeline.steps]))
    mlflow.log_param("feature_set", best_fs)
    mlflow.log_param("n_features_used", len(BEST_FEATURES))
    mlflow.log_param("training_rows", len(X_full_raw))

    mlflow.log_metric("val_wmae", val_wmae)

    signature = infer_signature(X_val_raw.head(100), val_preds[:100])

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="lightgbm_pipeline",
        signature=signature,
        input_example=X_val_raw.head(5),
        registered_model_name="walmart_lightgbm",
        serialization_format="cloudpickle"
    )

    mlflow.set_tag("stage", "final")
    mlflow.set_tag("model_type", "LightGBM")

    print(f"\nFinal Pipeline trained")
    print(f"Val WMAE: {val_wmae:.2f}")
    print(f"Registered as 'walmart_lightgbm' in Model Registry")
    print(f"Run ID: {final_run.info.run_id}")


## 13. Pipeline-ის ვალიდაცია raw test-ზე

In [ ]:
test_predictions = final_pipeline.predict(test_raw)

print(f"Raw test shape: {test_raw.shape}")
print(f"Predictions shape: {test_predictions.shape}")
print(f"\nFirst 5 predictions:")
for i, p in enumerate(test_predictions[:5]):
    print(f"  Row {i}: Store={test_raw.iloc[i]['Store']}, "
          f"Dept={test_raw.iloc[i]['Dept']}, "
          f"Date={test_raw.iloc[i]['Date']}, "
          f"Predicted Weekly_Sales = {p:.2f}")

import joblib
joblib.dump(final_pipeline, f'{MODELS_DIR}/lightgbm_pipeline.pkl')
print(f"\nPipeline saved: {MODELS_DIR}/lightgbm_pipeline.pkl")


## 14. Feature Importance

In [ ]:
lgb_model = final_pipeline.named_steps['model']

importances = pd.DataFrame({
    'feature': BEST_FEATURES,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(importances['feature'].head(20)[::-1], importances['importance'].head(20)[::-1])
plt.title('LightGBM Top 20 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(importances.head(10).to_string(index=False))


## 15. შეჯამება

LightGBM-ის pipeline გაეშვა იმავე ძირითადი MLflow run-ის სტრუქტურით — dagshub/Drive კავშირი უცვლელია. დამატებულია lag/rolling და `store_dept_median`/`roll_max_4w` ფიჩერები, დეტალური EDA, 20-კომბინაციიანი Grid Search (`num_leaves` x `learning_rate`), ცალკე `LightGBM_EarlyStopping_Tuning` run ზუსტი `n_estimators`-ის მოსაძებნად, და მოდელი ვწვრთნით `log1p(Weekly_Sales)`-ზე — პროგნოზები საბოლოოდ `expm1`-ით ბრუნდება საწყის მასშტაბში.

---

### შედეგები

| ეტაპი | შედეგი |
|---|---|
| **Feature Selection** | `lag_rolling_onehot` ნაკრები დანიშნულებით გამოიყენება downstream-ში |
| **5-fold TimeSeriesSplit CV** | ეტაპობრივი ვალიდაცია baseline პარამეტრებზე |
| **Grid Search (20 კომბინაცია)** | `num_leaves` x `learning_rate` მოძებნილია TimeSeriesSplit CV-ზე |
| **Early Stopping Tuning** | ოპტიმალური `n_estimators` მოძებნილია ვალიდაციაზე (upper bound 3000) |
| **Model Registry** | დარეგისტრირდა `walmart_lightgbm` ახალი ვერსია |

რეალური რიცხვები (`val_wmae`, `best_cv_wmae`) დამოკიდებულია რეალურ Walmart მონაცემებზე გაშვების შედეგად და ჩაიწერება MLflow-ში ავტომატურად — ისინი ამ notebook-ის კოდში წინასწარ არ არის დაფიქსირებული.